In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [1]:
from pyspark.sql import SparkSession
import re

In [2]:
import os
os.environ['PYSPARK_PYTHON']='python'

In [3]:
spark = SparkSession.builder.appName("word_count").getOrCreate()

25/12/14 10:18:57 WARN Utils: Your hostname, Biplovs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.12.38.73 instead (on interface en0)
25/12/14 10:18:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/14 10:18:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/14 10:18:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
sc = spark.sparkContext

In [5]:
text = """ Big Data Fun
Big Data Complex
Hadoop core component Big Data Analytics
"""

In [6]:
from pyspark.sql.functions import explode, split, lower, col, count

In [7]:
df = spark.createDataFrame([(text,)], ['text'])

In [8]:
df.show()

+--------------------+
|                text|
+--------------------+
| Big Data Fun\nBi...|
+--------------------+



In [9]:
# spliiting the words in to row
words_df = (df.select(explode(split(lower(col('text')),r'[\s\W]+')).alias('word')).filter(col('word') != ''))

# here \s is whute space and r denote regular expresion
# \W symbols punctuation and non words
# + grouping

In [10]:
words_df.show()

+---------+
|     word|
+---------+
|      big|
|     data|
|      fun|
|      big|
|     data|
|  complex|
|   hadoop|
|     core|
|component|
|      big|
|     data|
|analytics|
+---------+



In [11]:
words_count_df = words_df.groupBy('word').count().orderBy(col('count').desc())
words_count_df.show()

[Stage 6:>                                                        (0 + 10) / 10]

+---------+-----+
|     word|count|
+---------+-----+
|     data|    3|
|      big|    3|
|  complex|    1|
|component|    1|
|     core|    1|
|      fun|    1|
|analytics|    1|
|   hadoop|    1|
+---------+-----+



## method 2 using RDD

In [12]:
rdd = sc.parallelize([text])

In [13]:
words_rdd = rdd.flatMap(lambda line: re.findall(r'\b\w+\b', line.lower()))

In [14]:
pairs_rdd = words_rdd.map(lambda word: (word,1))

In [15]:
word_counts_rdd = pairs_rdd.reduceByKey(lambda a, b: a+b)

In [20]:
sorted_counts = word_counts_rdd.sortBy(lambda x: x[1], ascending = False)


In [26]:
!pip3 install psutil
 # we are installing it here because even if it is installed in anaconda it is not working here so


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [27]:
# or u can simoply use sorted_counts.collect()
for word, count in sorted_counts.collect():
    print(f"{word}:{count}")

big:3
data:3
complex:1
fun:1
core:1
hadoop:1
analytics:1
component:1
